# Solution F1 · OLS, Ridge, Lasso, ElasticNet: which one wins, and when

**Type**: Compare

## Objective

Compare four linear estimators on four controlled scenarios (number of observations, number of features,
correlation, sparsity) with a repeated simulation, and understand from the numbers, not from the theory, in which scenario
each one wins.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import ElasticNetCV, LassoCV, LinearRegression, RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def generate(n, p, rho, k, seed):
    rng = np.random.default_rng(seed)
    C = rho ** np.abs(np.subtract.outer(np.arange(p), np.arange(p)))       # correlation rho^|i-j|
    X = rng.normal(size=(n, p)) @ np.linalg.cholesky(C).T
    beta = np.zeros(p)
    beta[:k] = rng.choice([-1, 1], k) * rng.uniform(1, 3, k)
    return X, X @ beta + rng.normal(size=n) * 3


SCENARIOS = {"S1": dict(n=200, p=10, rho=0.0, k=10), "S2": dict(n=50, p=40, rho=0.0, k=5),
             "S3": dict(n=100, p=20, rho=0.9, k=20), "S4": dict(n=50, p=40, rho=0.7, k=5)}


def models():
    return {
        "ols": LinearRegression(),
        "ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 30))),
        "lasso": make_pipeline(StandardScaler(), LassoCV(cv=3, max_iter=5000, random_state=0)),
        "enet": make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.3, 0.7, 0.95], cv=3, max_iter=5000, random_state=0)),
    }


def coef_of(model):
    """Coefficients of the model, even if it sits inside a Pipeline."""
    last = model[-1] if hasattr(model, "steps") else model
    return last.coef_

## Commented solution

In [ ]:
def evaluate(scenario_name, repetitions=15):
    par = SCENARIOS[scenario_name]
    rows = []
    for rep in range(repetitions):
        X, y = generate(par["n"] + 2000, par["p"], par["rho"], par["k"], seed=rep * 10 + 1)
        X_tr, y_tr = X[:par["n"]], y[:par["n"]]
        X_te, y_te = X[par["n"]:], y[par["n"]:]                # 2000 test points: very precise error estimate
        for model_name, m in models().items():
            m.fit(X_tr, y_tr)
            rows.append({
                "scenario": scenario_name,
                "model": model_name,
                "rmse": mean_squared_error(y_te, m.predict(X_te), squared=False),
                "n_nonzero": int((coef_of(m) != 0).sum()),
            })
    return pd.DataFrame(rows)

In [ ]:
results = pd.concat([evaluate(name) for name in SCENARIOS], ignore_index=True)
table = results.pivot_table(index="scenario", columns="model", values="rmse", aggfunc="mean")
print("Mean RMSE on the test set (the theoretical minimum is 3.0, the standard deviation of the noise):")
table.round(2)

In [ ]:
winner = table.idxmin(axis=1).to_dict()
nonzero = results.pivot_table(index="scenario", columns="model", values="n_nonzero", aggfunc="mean")
print("winner:", winner)
print("mean number of nonzero coefficients (the true features are 10, 5, 20, 5):")
nonzero.round(1)

In [ ]:
true_false = {"a": True, "b": True, "c": False}

## Check

In [ ]:
with verify():
    check(len(results) == 4 * 15 * 4, "results has one row per scenario, repetition and model",
          "you need 4 scenarios x 15 repetitions x 4 models = 240 rows")
    check(list(table.index) == ["S1", "S2", "S3", "S4"] and set(table.columns) == {"ols", "ridge", "lasso", "enet"},
          "the table has scenarios on the rows and models on the columns", "use pivot_table(index='scenario', columns='model', values='rmse')")
    check(table.loc["S1"].max() / table.loc["S1"].min() < 1.03,
          "in S1 the four methods are equivalent (difference < 3%)", "in S1 (large n, small p) the methods should give the same RMSE")
    check(all(table.loc[s, "ols"] > 1.4 * table.loc[s].min() for s in ["S2", "S4"]),
          "in S2 and S4 OLS is much worse than the best (at least 1.4 times)",
          "with p close to n OLS should have a much higher RMSE than the best")
    check(winner == table.idxmin(axis=1).to_dict(), "winner consistent with the table", "winner = table.idxmin(axis=1).to_dict()")
    check(winner["S2"] in ("lasso", "enet") and winner["S4"] in ("lasso", "enet"),
          "in S2 and S4 the methods that zero the coefficients win", "in S2 and S4 the winner should be lasso or enet")
    check(nonzero.loc["S2", "ols"] == 40 and nonzero.loc["S2", "ridge"] == 40,
          "OLS and Ridge never zero anything", "OLS and Ridge must have all 40 coefficients nonzero")
    check(nonzero.loc["S2", "lasso"] < 30, "the Lasso in S2 keeps many fewer than 40 features",
          "the Lasso should zero a good part of the 40 coefficients in S2")
    check(true_false == {"a": True, "b": True, "c": False}, "statements judged correctly",
          "look at table and nonzero: Ridge zeroes nothing")
done()

## What you learned

- **S1** (n=200, p=10): the estimation noise is already small, and regularization has nothing to gain: all the
  methods reach an RMSE close to the theoretical minimum (3.0). Regularizing does no harm, but it is not needed.
- **S2 and S4** (p close to n, sparse effects): OLS has 41 parameters to estimate with 50 observations and pays an enormous
  variance (RMSE about 6.6, double the noise). Lasso and ElasticNet win because they zero the useless coefficients, as the
  true structure requires; Ridge shrinks but does not zero and stays in the middle.
- **S3** (correlation 0.9, all effects true): with n=100 and p=20 there is not much to gain and the differences are
  small: the rule "high correlation → ElasticNet" is not readable from the numbers. Repeating the scenarios with more
  repetitions serves exactly to avoid seeing differences where there are none.
- The Lasso still keeps about 14 features in S2 instead of the 5 true ones: it selects to **predict well**, not to
  reconstruct the truth (see exercise B2).

## Common pitfall

Concluding from a single repetition (or from a single real dataset) that "method X is the best". The differences
between methods are often of the same order as the variability from one sample to another: without many repetitions and a
large test set you see differences that do not exist.

## Interview question

<details><summary>You have 300 observations, 60 features correlated with each other and you don't know how many matter. Ridge, Lasso or ElasticNet?</summary>

I would not choose a priori: I would compare them in nested cross-validation, because they are three different hypotheses about the world. Ridge
assumes that many features contribute a little, Lasso that very few matter, ElasticNet is the compromise and copes better with
correlated groups (the Lasso picks one at random per group). With highly correlated features I would start from ElasticNet or
Ridge, and I would use the stability of the selection on bootstrap samples if I had to interpret the chosen features.

</details>